In [1]:
import os
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
DATE_START = pd.Timestamp("2022-07-01 00:00:00")
DATE_END   = pd.Timestamp("2025-06-30 23:00:00")

# Raw (pre-imputation) directories from config.yaml
datasets = {
    "CPCB (India)":      "/home/rishi/ML Projects/Air Pollution/CPCB/separated",
    "EPA (USA)":         "/home/rishi/ML Projects/Air Pollution/EPA/epa_data_by_site_all_years",
    "AURN (UK)":         "/home/rishi/ML Projects/Air Pollution/AURN/aurn_processed",
    "EEA FR (France)":   "/home/rishi/ML Projects/Air Pollution/EEA/FR/processed",
    "EEA DE (Germany)":  "/home/rishi/ML Projects/Air Pollution/EEA/DE/processed",
    "SINAICA (Mexico)":  "/home/rishi/ML Projects/Air Pollution/SINAICA/processed",
    "CNEMC (China)":     "/home/rishi/ML Projects/Air Pollution/CNEMC/separated",
}

In [3]:
def _count_file(filepath, date_start, date_end):
    try:
        df = pd.read_csv(filepath, parse_dates=["Timestamp"])
    except (ValueError, KeyError):
        return 0, 0, None
    mask = (df["Timestamp"] >= date_start) & (df["Timestamp"] <= date_end)
    df = df.loc[mask]
    pollutant_col = df.columns[1]
    # Extract site id: filename stem without the trailing _POLLUTANT suffix
    stem = filepath.stem  # e.g. "site_103_CRRI_Mathura_Road_Delhi_IMD_CO"
    site_id = "_".join(stem.rsplit("_", 1)[:-1])  # drop last "_CO" / "_NO2" etc.
    return len(df), int(df[pollutant_col].notna().sum()), site_id


def count_dataset(directory, date_start, date_end, max_workers=24):
    csv_files = sorted(Path(directory).glob("*.csv"))
    total = 0
    non_nan = 0
    sites = set()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_count_file, f, date_start, date_end): f for f in csv_files}
        for future in tqdm(as_completed(futures), total=len(futures), desc=Path(directory).name, leave=False):
            t, nn, site_id = future.result()
            total   += t
            non_nan += nn
            if site_id is not None:
                sites.add(site_id)

    return total, non_nan, len(sites)

In [ ]:
rows = []
for name, directory in datasets.items():
    if not os.path.isdir(directory):
        print(f"[SKIP] Directory not found: {directory}")
        continue
    print(f"Processing {name} ...")
    total, non_nan, n_sites = count_dataset(directory, DATE_START, DATE_END)
    rows.append({
        "Dataset":          name,
        "Unique Sites":     n_sites,
        "Total Points":     total,
        "Non-NaN Points":   non_nan,
        "NaN Points":       total - non_nan,
        "Completeness (%)": round(100 * non_nan / total, 2) if total > 0 else 0.0,
    })
    print(f"  sites={n_sites:,}  total={total:,}  non-nan={non_nan:,}")

Processing CPCB (India) ...


separated:   0%|          | 0/3323 [00:00<?, ?it/s]

In [ ]:
df_results = (
    pd.DataFrame(rows)
    .sort_values("Dataset")
    .reset_index(drop=True)
)

df_display = df_results.copy()
for col in ["Unique Sites", "Total Points", "Non-NaN Points", "NaN Points"]:
    df_display[col] = df_display[col].apply(lambda x: f"{x:,}")

df_display.style.set_caption(f"Data point counts per dataset  |  {DATE_START.date()} – {DATE_END.date()}")

,Dataset,Unique Sites,Total Points,Non-NaN Points,NaN Points,Completeness (%)
0,AURN (UK),203,"15,808,704","12,687,536","3,121,168",80.260000
1,CNEMC (China),"1,728","272,719,872","253,839,247","18,880,625",93.080000
2,CPCB (India),564,"83,447,280","63,230,742","20,216,538",75.770000
3,EEA DE (Germany),446,"43,375,296","38,660,413","4,714,883",89.130000
4,EEA FR (France),564,"40,455,552","32,333,821","8,121,731",79.920000
5,EPA (USA),"2,006","105,742,080","83,472,700","22,269,380",78.940000
6,SINAICA (Mexico),169,"23,384,256","11,491,049","11,893,207",49.140000


In [ ]:
df_results.sum(axis=0)

Dataset             AURN (UK)CNEMC (China)CPCB (India)EEA DE (Germ...
Unique Sites                                                     5680
Total Points                                                584933040
Non-NaN Points                                              495715508
NaN Points                                                   89217532
Completeness (%)                                               546.24
dtype: object

In [ ]:
df_results['Non-NaN Points'].sum()/df_results['Total Points'].sum()

0.8474739399231064